# Coloration de cartes — Recherche Tabou hybride ML
**M1 MIAGE** — [Prénom NOM 1], [Prénom NOM 2], [Prénom NOM 3]

Ce notebook est le **point d'entrée unique**. Tout le code métier est dans `src/`.
Modifier le code → modifier les fichiers `.py`, pas ce notebook.

---
## 1. Imports & configuration

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# Import de tout le code métier
from src.graph import (
    generate_map_graph, generate_dsjc_like,
    count_conflicts, objective, initial_solution,
    solution_info, visualize_coloring
)
from src.tabu import tabu_search
from src.ml_selector import collect_training_data, train_model, evaluate_model

SEED     = 42
N_COLORS = 4
MAX_ITER = 500
N_RUNS   = 10

random.seed(SEED)
np.random.seed(SEED)
print('✅ Imports OK')

---
## 2. Génération des instances

In [ ]:
G_small  = generate_map_graph(n_regions=15, seed=42)
G_medium = generate_map_graph(n_regions=30, seed=43)
G_large  = generate_map_graph(n_regions=50, seed=44)
G_dsjc   = generate_dsjc_like(n=50, p=0.3, seed=42)

import networkx as nx
for name, G in [('G_small', G_small), ('G_medium', G_medium),
                ('G_large', G_large),  ('G_dsjc',   G_dsjc)]:
    print(f'{name:10s} | nœuds={G.number_of_nodes():3d} '
          f'| arêtes={G.number_of_edges():4d} '
          f'| densité={nx.density(G):.3f}')

# Visualisation initiale
sol_greedy = nx.coloring.greedy_color(G_small, strategy='largest_first')
visualize_coloring(G_small, sol_greedy, 'Coloration greedy initiale (G_small)')

---
## 2-bis. Carte réelle de France (optionnel)

In [ ]:
# Décommenter pour utiliser la vraie carte de France
# pip install geopandas shapely requests

# from src.france_map import load_france_graph, visualize_france
# gdf, G_france = load_france_graph()
# sol_greedy_france = nx.coloring.greedy_color(G_france, strategy='largest_first')
# visualize_france(gdf, G_france, sol_greedy_france, 'Greedy — France')

---
## 3. Tabou de base (baseline)

In [ ]:
result_base = tabu_search(
    G_medium, n_colors=N_COLORS, max_iter=MAX_ITER,
    tabu_tenure=10, operator_mode='random', verbose=True
)
print('\nRésultat baseline (random) :')
solution_info(G_medium, result_base['best_coloring'])
visualize_coloring(G_medium, result_base['best_coloring'], 'Tabou Random — G_medium')

---
## 4. Collecte des données ML

In [ ]:
train_graphs = [
    generate_map_graph(n_regions=n, seed=s)
    for n, s in [(15,1),(20,2),(25,3),(30,4),(35,5),(40,6)]
]

t0 = time.time()
X_raw, y_raw = collect_training_data(
    train_graphs, n_colors=N_COLORS, n_runs_per_graph=3, max_iter=200)
print(f'Terminé en {time.time()-t0:.1f}s — {len(X_raw)} échantillons')
print(f'recolor={sum(y_raw==0)} | swap={sum(y_raw==1)} | kempe={sum(y_raw==2)}')

---
## 5. Entraînement du Random Forest

In [ ]:
rf_model, scaler, df_features = train_model(
    X_raw, y_raw, train_graphs=train_graphs, n_colors=N_COLORS)
evaluate_model(rf_model, scaler, df_features)

---
## 6. Comparaison Random vs ML

In [ ]:
modes       = ['random', 'recolor', 'swap', 'kempe', 'ml']
results_all = {m: [] for m in modes}

for mode in modes:
    for run in range(N_RUNS):
        random.seed(run); np.random.seed(run)
        res = tabu_search(
            G_medium, n_colors=N_COLORS, max_iter=MAX_ITER, tabu_tenure=10,
            operator_mode=mode,
            ml_model=rf_model if mode=='ml' else None,
            scaler=scaler     if mode=='ml' else None,
        )
        results_all[mode].append({
            'conflicts': res['best_conflicts'],
            'obj'      : res['best_obj'],
            'n_colors' : res['best_n_colors'],
        })
    avg_c = np.mean([r['conflicts'] for r in results_all[mode]])
    print(f'{mode:8s} → conflits_moy={avg_c:.2f}')

---
## 7. Courbes de convergence

In [ ]:
random.seed(SEED); np.random.seed(SEED)
colors_mode  = {'random': 'gray', 'ml': 'crimson', 'kempe': 'steelblue'}
conv_results = {}

for mode in ['random', 'ml', 'kempe']:
    conv_results[mode] = tabu_search(
        G_medium, n_colors=N_COLORS, max_iter=MAX_ITER, tabu_tenure=10,
        operator_mode=mode,
        ml_model=rf_model if mode=='ml' else None,
        scaler=scaler   if mode=='ml' else None,
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for mode in ['random', 'ml', 'kempe']:
    hist = conv_results[mode]['history']
    axes[0].plot(hist['obj'],       label=mode, color=colors_mode[mode], alpha=0.8)
    axes[1].plot(hist['conflicts'], label=mode, color=colors_mode[mode], alpha=0.8)

for ax, ylabel in zip(axes, ['Valeur objectif', 'Nombre de conflits']):
    ax.set_xlabel('Itération'); ax.set_ylabel(ylabel); ax.legend()

axes[0].set_title('Convergence — Objectif')
axes[1].set_title('Convergence — Conflits')
plt.tight_layout()
plt.show()

---
## 8. Résumé final

In [ ]:
rows = []
for mode in modes:
    data = results_all[mode]
    rows.append({
        'Mode'            : mode,
        'Conflits moy.'   : np.mean([r['conflicts'] for r in data]),
        'Conflits std.'   : np.std([r['conflicts']  for r in data]),
        'Conflits min.'   : np.min([r['conflicts']  for r in data]),
        'Objectif moy.'   : np.mean([r['obj']       for r in data]),
    })

df_results = pd.DataFrame(rows).set_index('Mode')
display(df_results.round(3))